# Temporal Vehicle-Tier GRU — 3-Tier Federated Model (cleaned)

**One of the 5 single-format ML notebooks.** This is the vehicle/OBU temporal
model. It trains **only** the 3-tier hierarchical FL variant
(OBU → serving-RSU → SDN-controller → global) — no centralized baseline,
no single-hop FL, no sensitivity sweep.

It obeys the shared contract every ML notebook follows:

| item | value |
|---|---|
| **input** | `seq6_pct_sweep_*/pct*_s1/communication_log.csv` (multi-run, tagged `run_id`) |
| **window** | count-based **W = 10 beacons**, step 1 (paper §4.2.2) |
| **stamp** | every window carries `window_start_seconds` = time of its first beacon |
| **key** | `claimed_node_id, window_start_seconds, split` |
| **labels** | 7-class `attack_type ∈ {0..6}` + `is_sybil ∈ {0,1}` |
| **split** | shared `fl/artifacts/split_map.parquet` (group-stratified by identity) |
| **output** | `fl/artifacts/temporal_vehicle_tier.parquet` = `[key | labels | phi_temp_0..31 | p_temp_0..6]` |

7 classes: `0 legit, 1 outsider, 2 sim, 3 nonsim, 4 indirect, 5 malicious_rsu, 6 malicious_controller`.
The vehicle tier can only really observe classes 0–4; slots 5/6 stay in the output
vector (mostly empty) so shapes match the RSU tier and the fusion stage.

In [1]:
import os, glob, json, warnings, random, copy, time
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED); random.seed(RANDOM_SEED)

BASE_DIR = "/home/sdvn_defense_sybil/35_sybil_attack_project/ns-allinone-3.35/ns-3.35/sybil-attack"

# ── Shared fusion spine (single source of truth for the 3 vehicle-tier models):
#    identical identity subsample, leakage-safe split, and 2 s export grid so the
#    phi tables join on (run_id, claimed_node_id, window_start_seconds, split).
import sys; sys.path.insert(0, f"{BASE_DIR}/ml/fusion")
import identity_manifest as IDM
MAN = IDM.load_manifest()          # builds notebooks/outputs/split_map.parquet once

# ── INPUT: the three focus runs come from the shared spine ──
RUN_DIRS = IDM.RUN_DIRS
assert RUN_DIRS, "no pct*_s1 run dirs found — check identity_manifest.RUN_DIRS"

# ── OUTPUT: shared artifacts dir (all 3 vehicle-tier notebooks read/write here) ──
ARTIFACTS_DIR  = str(IDM.ARTIFACTS_DIR)
SPLIT_MAP_PATH = str(IDM.SPLIT_MAP_PATH)   # owned/built by identity_manifest
OUT_PARQUET    = os.path.join(ARTIFACTS_DIR, "temporal_vehicle_tier.parquet")
MODEL_DIR      = os.path.join(ARTIFACTS_DIR, "temporal_vehicle_tier")
os.makedirs(MODEL_DIR, exist_ok=True)

RSU_POS_FILE = os.path.join(BASE_DIR, "inputs/mobility/kuala-lumpur-bb/klbb2km_rsus_8x8.csv")

# ── 7-class taxonomy (index == attack_type == attack_type_label) ──
ATTACK_CLASSES = IDM.ATTACK_CLASSES
N_CLASSES = len(ATTACK_CLASSES)
PHASE_COL = "attack_type_label"   # seq6/mode-9: active-phase type 1..6 (attack_type col is a dead const)

# ── Window (paper §4.2.2) ──
WINDOW_W    = 10    # beacons per window (count-based); sweep later over {5,10,20}
WINDOW_STEP = 1
SEQ_MIN_LEN = 3
EXPORT_GRID_S = IDM.EXPORT_GRID_S  # shared 2 s fusion grid

# HPC memory guard: cap total windows so X_all + scaled copies fit in RAM (see preflight).
# ~69M windows would need >50 GB; 18M keeps peak ~15 GB and is still ample training data.
MAX_WINDOWS = 18_000_000

# ── Tractability: the SHARED identity subsample (same ids for all 3 models) ──
# comes from IDM.load_manifest(); IDENTITY_KEEP_FRAC lives in identity_manifest.KEEP_FRAC

# ── GRU (paper Table 4.6) ──
GRU_HIDDEN, GRU_DROPOUT = 32, 0.3

# ── FL topology + fixed 3-tier hyperparameters (no sweep here) ──
N_RSUS, N_CONTROLLERS = 64, 4
FL_PARAMS = dict(fl_rounds=250, client_ratio=0.3, local_epochs=3,
                 local_lr=0.001, mu=0.01, dp_sigma=0.1)
PARTICIPATION = "fuzzy"     # OBU client selection (paper §4.10.2)
FL_TRIM_FRAC, FL_ES_PATIENCE = 0.10, 5

USE_COLS = ["receive_time","flow","receiver_role","receiver_id","real_node_id",
            "claimed_node_id","message_type","sequence_number","packet_size","delay",
            "bsm_temporary_id","bsm_msg_count","bsm_x","bsm_y","bsm_speed","bsm_heading",
            "attack_type","attack_type_label"]

import torch
torch.manual_seed(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device {DEVICE}")
print(f"{len(RUN_DIRS)} runs:", [os.path.basename(d) for d in RUN_DIRS])
print(f"W={WINDOW_W} beacons | {N_CLASSES} classes | export grid {EXPORT_GRID_S}s")


torch 2.8.0+cu128 | device cuda
3 runs: ['pct100_s1', 'pct20_s1', 'pct40_s1']
W=10 beacons | 7 classes | export grid 2.0s


## 1. Load & label beacons (7-class, multi-run)

Per beacon: `attack_type = attack_type_label` when the claimed id ≠ real sender,
else `0`. `is_sybil = attack_type > 0`. Nothing is dropped — types 5/6 keep their
label. Each run is tagged with `run_id` and identity-subsampled independently.

In [2]:
def load_single_run(run_dir):
    comm = os.path.join(run_dir, "communication_log.csv")
    run_id = os.path.basename(run_dir)
    # Memory-safe: stream the ~12 GB comm log in chunks, keeping ONLY the shared
    # identity subsample. The full file is never materialised (fixes the prior stall).
    df = IDM.read_run_log_filtered(comm, (lambda c: c in USE_COLS),
                                   "claimed_node_id", run_id, MAN)
    assert PHASE_COL in df.columns, f"{PHASE_COL} missing in {comm}"
    mismatch = (df["real_node_id"] != df["claimed_node_id"]).values
    df["attack_type"] = np.where(mismatch, df[PHASE_COL].astype(int), 0).astype("int8")  # 0..6
    df["is_sybil"]    = (df["attack_type"] > 0).astype("int8")
    df["run_id"]      = run_id
    return df

raw_df = pd.concat([load_single_run(d) for d in RUN_DIRS], ignore_index=True)
print("Raw rows:", f"{len(raw_df):,}")
print(raw_df.groupby("run_id")["is_sybil"].agg(["size","mean"]))
print("\nPer-beacon class counts:")
print(raw_df["attack_type"].value_counts().sort_index().rename(index=dict(enumerate(ATTACK_CLASSES))))


    [safe-read] communication_log.csv: scanned 55,648,149 rows, kept 28,164,610 (50.6%)


    [safe-read] communication_log.csv: scanned 64,372,236 rows, kept 26,675,035 (41.4%)


    [safe-read] communication_log.csv: scanned 62,635,678 rows, kept 23,698,468 (37.8%)


Raw rows: 78,538,113


               size      mean
run_id                       
pct100_s1  28164610  0.114787
pct20_s1   26675035  0.080609
pct40_s1   23698468  0.104519

Per-beacon class counts:
attack_type
legitimate              70677988
outsider                 2450123
sim                      3986387
nonsim                    809235
indirect                  527623
malicious_rsu              82983
malicious_controller        3774
Name: count, dtype: int64


## 2. Vehicle-tier filter & deduplicate

Keep only V2V beacons received by vehicles. Deduplicate to one row per beacon
event (multiple RSUs/OBUs may hear the same beacon) and attach `observer_count`.

In [3]:
BEACON_KEY = ["run_id","real_node_id","claimed_node_id","bsm_temporary_id","receive_time"]

def dedup_beacons(df):
    dup = (df.groupby(BEACON_KEY)["receiver_id"].nunique().rename("observer_count").reset_index())
    agg_cols = [c for c in df.columns if c not in ["receiver_id","receiver_role"]]
    b = df.sort_values("receive_time").groupby(BEACON_KEY, as_index=False)[agg_cols].first()
    return b.merge(dup, on=BEACON_KEY, how="left")

veh_raw = raw_df[(raw_df["flow"]=="v2v_beacon") & (raw_df["receiver_role"]=="vehicle")].copy()
v2v = dedup_beacons(veh_raw)
# per-row view keeps receiver_id (needed for windowing + FL client assignment)
v2v_raw = veh_raw.merge(v2v[BEACON_KEY + ["observer_count"]], on=BEACON_KEY, how="left")
del raw_df, veh_raw
assert set(v2v["flow"].unique()) == {"v2v_beacon"}
print("beacon events:", f"{len(v2v):,}", "| per-row:", f"{len(v2v_raw):,}")
print("claimed identities:", v2v["claimed_node_id"].nunique(), "| OBUs:", v2v_raw["receiver_id"].nunique())

import gc; gc.collect()   # reclaim the freed raw beacons (HPC RAM)


beacon events: 1,369,488 | per-row: 76,211,686
claimed identities: 1172 | OBUs: 259


20

## 3. Build W=10 windows & stamp `window_start_seconds`

Sliding windows of 10 beacons per `(run_id, receiver_id, claimed_node_id)` stream.
Each window records `window_start_seconds` = the receive time of its first beacon,
and its 7-class label (`attack_type` = majority type among sybil beacons, else 0).

In [4]:
SEQ_FEATURE_COLS = ["delta_t","temp_id_change_flag","bsm_x","bsm_y","bsm_speed",
                    "bsm_heading","packet_size","delay","observer_count","bsm_msg_count_delta"]
assert "real_node_id" not in SEQ_FEATURE_COLS

def build_windows(df, W=WINDOW_W, step=WINDOW_STEP, min_len=SEQ_MIN_LEN, keep_prob=1.0):
    Xs, Ls, ys, meta = [], [], [], []
    _wr = np.random.default_rng(RANDOM_SEED)
    for (run, rid, cid), g in df.sort_values("receive_time").groupby(
            ["run_id","receiver_id","claimed_node_id"], sort=False):
        g = g.reset_index(drop=True)
        if len(g) < min_len: continue
        delta_t   = g["receive_time"].diff().fillna(0.0).values
        msg_delta = g["bsm_msg_count"].diff().fillna(0.0).values
        tid_chg   = (g["bsm_temporary_id"] != g["bsm_temporary_id"].shift(1)).fillna(False).astype(float).values
        feat = np.column_stack([delta_t, tid_chg, g["bsm_x"].values, g["bsm_y"].values,
                                g["bsm_speed"].values, g["bsm_heading"].values,
                                g["packet_size"].values, g["delay"].values,
                                g["observer_count"].values, msg_delta]).astype(np.float32)
        t   = g["receive_time"].values
        at  = g["attack_type"].values
        syb = g["is_sybil"].values
        n = len(feat)
        for s in range(0, n - W + 1, step):
            if keep_prob < 1.0 and _wr.random() >= keep_prob:
                continue   # memory-budgeted window subsample (seeded, leakage-safe)
            e = s + W
            win, L = feat[s:e], W
            if L < min_len: continue
            padded = np.zeros((W, feat.shape[1]), np.float32); padded[:L] = win
            wt = at[s:e]; ws = syb[s:e]
            if ws.max() > 0:
                lab = int(pd.Series(wt[wt > 0]).mode().iloc[0]); iss = 1
            else:
                lab, iss = 0, 0
            Xs.append(padded); Ls.append(L); ys.append(lab)
            meta.append({"run_id": run, "receiver_id": rid, "claimed_node_id": cid,
                         "window_start_seconds": float(t[s]), "attack_type": lab, "is_sybil": iss})
    return (np.stack(Xs), np.array(Ls, np.int32), np.array(ys, np.int64), pd.DataFrame(meta))

_kp = min(1.0, MAX_WINDOWS / max(len(v2v_raw), 1))
X_all, L_all, y_all, meta_df = build_windows(v2v_raw, keep_prob=_kp)
print(f"window keep_prob={_kp:.3f} (cap {MAX_WINDOWS:,})")
print("window tensor:", X_all.shape, "| windows:", f"{len(y_all):,}")
print("\nwindow class balance:")
print(pd.Series(y_all).value_counts().sort_index().rename(index=dict(enumerate(ATTACK_CLASSES))))

window keep_prob=0.236 (cap 18,000,000)
window tensor: (17927335, 10, 10) | windows: 17,927,335

window class balance:
legitimate       16227924
outsider           528784
sim                883654
nonsim             171716
indirect            99810
malicious_rsu       15447
Name: count, dtype: int64


## 4. Shared split (group-stratified by identity)

The **one** split every notebook reuses. Built here (first notebook to run) if it
doesn't exist yet: each `claimed_node_id` goes entirely to train/val/test 70:15:15,
stratified on whether that identity is ever sybil, so no identity leaks across splits.
Saved to `fl/artifacts/split_map.parquet` and imported by the other 4 notebooks.

In [5]:
# Split comes from the SHARED manifest (grouped by real vehicle, stratified by
# (attacker%, is-attacker); identical across RSSI / GRU / trust). We do NOT build
# our own split here — that was the cross-model inconsistency.
split_arr = IDM.split_of(meta_df["run_id"], meta_df["claimed_node_id"], MAN)
tr_m = (split_arr == "train"); va_m = (split_arr == "val"); te_m = (split_arr == "test")
X_tr,L_tr,y_tr = X_all[tr_m],L_all[tr_m],y_all[tr_m]
X_va,L_va,y_va = X_all[va_m],L_all[va_m],y_all[va_m]
X_te,L_te,y_te = X_all[te_m],L_all[te_m],y_all[te_m]

# leakage guard: no real vehicle may straddle splits (checked via the shared map)
_vk = MAN.set_index(["run_id","claimed_node_id"])["vehicle_key"]
_mk = list(zip(meta_df["run_id"], meta_df["claimed_node_id"]))
veh = np.array([_vk.get((r,c), f"{r}?{c}") for r,c in _mk], dtype=object)
assert not (set(veh[tr_m]) & set(veh[te_m])), "identity leak across train/test!"
print(f"train {tr_m.sum():,} | val {va_m.sum():,} | test {te_m.sum():,}")
print("windows per split match the shared map (by real vehicle).")


train 13,258,826 | val 2,066,457 | test 2,602,052
windows per split match the shared map (by real vehicle).


## 5. Scale on training timesteps only (leakage-safe)

In [6]:
from sklearn.preprocessing import StandardScaler
import pickle

def real_rows(X, L): return np.vstack([X[i,:L[i],:] for i in range(len(L)) if L[i] > 0])
scaler = StandardScaler().fit(real_rows(X_tr, L_tr))

def scale_seq(X, L, sc):
    out = X.copy()
    for i in range(len(L)):
        if L[i] > 0: out[i,:L[i],:] = sc.transform(X[i,:L[i],:])
    return out

X_tr_s, X_va_s, X_te_s = (scale_seq(X, L, scaler) for X,L in
                          [(X_tr,L_tr),(X_va,L_va),(X_te,L_te)])
pickle.dump(scaler, open(os.path.join(MODEL_DIR,"scaler.pkl"),"wb"))
print("scaled; scaler saved to", MODEL_DIR)

scaled; scaler saved to /home/sdvn_defense_sybil/35_sybil_attack_project/ns-allinone-3.35/ns-3.35/sybil-attack/notebooks/outputs/temporal_vehicle_tier


## 6. GRU model (7-class) & eval helpers

Unidirectional GRU(32) → Dropout → Linear(32→7). The 32-d hidden state is
`phi_temp` (exported for fusion); the 7-way softmax is `p_temp`. Trained with
class-weighted cross-entropy. Full-set inference is chunked to bound GPU memory.

In [7]:
import torch.nn as nn
from sklearn.metrics import matthews_corrcoef, f1_score, accuracy_score
N_FEATURES = X_tr_s.shape[2]

class TemporalGRU(nn.Module):
    def __init__(self, n_feat=N_FEATURES, hidden=GRU_HIDDEN, n_classes=N_CLASSES, dropout=GRU_DROPOUT):
        super().__init__()
        self.gru  = nn.GRU(n_feat, hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, n_classes)
    def forward(self, x, lengths, return_phi=False):
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.gru(packed)
        phi   = h_n[-1]                       # (B, 32) = phi_temp
        logits = self.head(self.drop(phi))    # (B, 7)
        return (logits, phi) if return_phi else logits

def to_tensors(X, L): return torch.tensor(X, dtype=torch.float32), torch.tensor(L, dtype=torch.long)
EVAL_BATCH = 65536

@torch.no_grad()
def _fwd(model, X, L, return_phi=False, batch=EVAL_BATCH):
    model.eval(); logits, phis = [], []
    for i in range(0, len(L), batch):
        xb = torch.tensor(X[i:i+batch], dtype=torch.float32).to(DEVICE)
        lb = torch.tensor(L[i:i+batch], dtype=torch.long)
        out = model(xb, lb, return_phi=return_phi)
        if return_phi: logits.append(out[0].cpu()); phis.append(out[1].cpu())
        else:          logits.append(out.cpu())
    logit = torch.cat(logits).numpy()
    return (logit, torch.cat(phis).numpy()) if return_phi else logit

def _softmax(z): e = np.exp(z - z.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)

@torch.no_grad()
def gru_eval(model, X, L, y):
    pred = _fwd(model, X, L).argmax(1)
    return dict(mcc=matthews_corrcoef(y, pred),
                f1=f1_score(y, pred, average="macro", zero_division=0),
                acc=accuracy_score(y, pred), pred=pred)

@torch.no_grad()
def extract_phi(model, X, L):
    logit, phi = _fwd(model, X, L, return_phi=True)
    return phi, _softmax(logit)          # (N,32), (N,7)

counts = np.bincount(y_tr, minlength=N_CLASSES)
class_w = torch.tensor(counts.sum() / (N_CLASSES * np.maximum(counts, 1)), dtype=torch.float32).to(DEVICE)
print("features:", N_FEATURES, "| train per-class:", counts.tolist())

features: 10 | train per-class: [12116969, 329595, 599743, 120198, 79448, 12873, 0]


## 7. FL clients & OBU→RSU→controller topology

FL clients are OBUs (`receiver_id`) drawn from the **train** split only. Each OBU is
mapped to its nearest serving RSU (geographic, from the 8×8 grid) and each RSU to one
of 4 SDN controllers by NS-3 index band — this is the 3-tier hierarchy. A Mamdani
fuzzy system scores OBU suitability for client selection (paper §4.10.2).

In [8]:
# FL client = one physical OBU in ONE sim = (run_id, receiver_id). The three pct
# runs reuse receiver ids (receiver 5 in pct20 != receiver 5 in pct100), so a bare
# receiver_id would merge three different OBUs into one client and average their
# positions across sims. Compose the key with run_id — same rule as RSSI v2.
tr_meta = meta_df.loc[tr_m].reset_index(drop=True)
client_rids = sorted(set(zip(tr_meta["run_id"], tr_meta["receiver_id"].astype(int))))  # (run_id, rid)
_rid_arr = tr_meta["receiver_id"].astype(int).values
_run_arr = tr_meta["run_id"].values
client_map = {}
for key in client_rids:
    m = (_run_arr == key[0]) & (_rid_arr == key[1])
    client_map[key] = (X_tr_s[m], L_tr[m], y_tr[m])

# OBU -> nearest serving RSU (position averaged WITHIN its own run only)
OBU_TO_RSU = {}
try:
    rp = pd.read_csv(RSU_POS_FILE); rxy = rp[["x","y"]].values
    legit = v2v_raw[v2v_raw["is_sybil"] == 0]
    sent  = legit.groupby(["run_id","receiver_id"])[["bsm_x","bsm_y"]].mean()
    for key in client_rids:
        run, rid = key
        if (run, rid) in sent.index:
            px, py = sent.loc[(run, rid), "bsm_x"], sent.loc[(run, rid), "bsm_y"]
            d2 = (rxy[:,0]-px)**2 + (rxy[:,1]-py)**2
            OBU_TO_RSU[key] = int(rp["rsu_id"].iloc[int(np.argmin(d2))])
        else:
            OBU_TO_RSU[key] = rid % N_RSUS
    print(f"OBU->RSU map over {len(set(OBU_TO_RSU.values()))}/{N_RSUS} RSUs")
except Exception as e:
    OBU_TO_RSU = {key: key[1] % N_RSUS for key in client_rids}
    print("[warn] RSU pos file unavailable:", e)

def rsu_to_controller(rsu_id): return min(N_CONTROLLERS-1, int(rsu_id)*N_CONTROLLERS//N_RSUS)

# -- Mamdani fuzzy OBU suitability (pure numpy) --
def _tri(x,a,b,c):
    if x<=a or x>=c: return 0.0
    return (x-a)/(b-a) if x<b else (c-x)/(c-b)
def _lo(x): return _tri(x,-1,0,.5)
def _me(x): return _tri(x,0,.5,1)
def _hi(x): return _tri(x,.5,1,2)
_U = np.linspace(0,1,101)
_OL,_OM,_OH = (np.array([f(u) for u in _U]) for f in (_lo,_me,_hi))
def _suit(m):
    a,b,mo,f = m
    r_hi = max(min(_hi(a),_hi(b),_hi(f)), min(_hi(a),_hi(f)), min(_hi(b),_hi(mo)))
    r_me = max(_me(a),_me(b),_me(f),_me(mo)); r_lo = max(_lo(a),_lo(f),_lo(b))
    agg = np.maximum.reduce([np.minimum(r_hi,_OH),np.minimum(r_me,_OM),np.minimum(r_lo,_OL)])
    s = agg.sum(); return float((_U*agg).sum()/s) if s>0 else .5

last_seen = v2v_raw.groupby(["run_id","receiver_id"])["receive_time"].max()
t0, tspan = v2v_raw["receive_time"].min(), max(v2v_raw["receive_time"].max()-v2v_raw["receive_time"].min(),1e-6)
raw = {}
for key,(Xc,Lc,yc) in client_map.items():
    n = len(yc); bal = 1-abs(.5-(yc.mean() if n else 0))*2
    rr = np.vstack([Xc[i,:Lc[i],:] for i in range(len(Lc)) if Lc[i]>0]) if n else np.zeros((1,N_FEATURES))
    mob = rr[:,4].std()+rr[:,5].std()
    fr = (last_seen.get(key,t0)-t0)/tspan
    raw[key] = [n,bal,mob,fr]
arr = {j:np.array([raw[k][j] for k in client_rids]) for j in range(4)}
norm = {j:(arr[j]-arr[j].min())/((arr[j].max()-arr[j].min()) or 1) for j in range(4)}
client_fuzzy = {key: _suit([norm[j][i] for j in range(4)]) for i,key in enumerate(client_rids)}

def select_clients(rids, n_sel, strategy="fuzzy"):
    if strategy == "random": return random.sample(rids, n_sel)
    if strategy == "fuzzy":  return sorted(rids, key=lambda r: client_fuzzy[r], reverse=True)[:n_sel]
    raise ValueError(strategy)

for cc in range(N_CONTROLLERS):
    rids = [r for r in client_rids if rsu_to_controller(OBU_TO_RSU[r])==cc]
    print(f"controller {cc}: {len(rids):3d} OBUs")
print(f"FL clients (OBU x run): {len(client_rids)}")


OBU->RSU map over 36/64 RSUs


controller 0: 238 OBUs
controller 1: 342 OBUs
controller 2: 120 OBUs
controller 3:  74 OBUs
FL clients (OBU x run): 774


## 8. Train the 3-tier hierarchical FL model

Per round: OBU local FedProx+DP update → per-RSU size-weighted FedAvg →
per-controller size-weighted FedAvg → global trimmed mean over the 4 controllers.
This is the only training in the notebook.

In [9]:
MAX_LOCAL_WINDOWS, LOCAL_BATCH, VAL_EVAL_SIZE = 16384, 256, 1_000_000
_rng = np.random.RandomState(RANDOM_SEED)
_vi = (_rng.choice(len(y_va), VAL_EVAL_SIZE, replace=False) if len(y_va) > VAL_EVAL_SIZE else np.arange(len(y_va)))
X_va_e, L_va_e, y_va_e = X_va_s[_vi], L_va[_vi], y_va[_vi]

def client_update(gsd, X, L, y, epochs, mu, lr, dp_sigma):
    if len(y) > MAX_LOCAL_WINDOWS:
        sel = np.random.choice(len(y), MAX_LOCAL_WINDOWS, replace=False); X,L,y = X[sel],L[sel],y[sel]
    w = TemporalGRU().to(DEVICE); w.load_state_dict(gsd); w.train()
    opt = torch.optim.Adam(w.parameters(), lr=lr)
    ce  = nn.CrossEntropyLoss(weight=class_w)
    gref = {k: v.detach().clone() for k,v in gsd.items()}
    xb = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    lb = torch.tensor(L, dtype=torch.long); yb = torch.tensor(y, dtype=torch.long).to(DEVICE)
    n = len(y)
    for _ in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, LOCAL_BATCH):
            idx = perm[i:i+LOCAL_BATCH]; opt.zero_grad()
            loss = ce(w(xb[idx], lb[idx]), yb[idx])
            prox = sum(((p-gref[k])**2).sum() for k,p in w.named_parameters())
            (loss + (mu/2)*prox).backward(); opt.step()
    new = w.state_dict(); up = {}
    for k in new:
        d = new[k]-gsd[k]
        if torch.is_floating_point(d): d = d + torch.randn_like(d)*dp_sigma
        up[k] = gsd[k]+d
    return up, n

def w_fedavg(sds, ws):
    tot = float(sum(ws)) or 1.0
    return {k: sum(sd[k].float()*(w/tot) for sd,w in zip(sds,ws)) for k in sds[0]}

def trimmed_mean(sds, frac):
    n = len(sds); k = int(frac*n); out = {}
    for key in sds[0]:
        st,_ = torch.sort(torch.stack([sd[key].float() for sd in sds],0),0)
        out[key] = st[k:n-k].mean(0) if k>0 and n-2*k>0 else st.mean(0)
    return out

def run_fl_hier(fl_rounds, client_ratio, local_epochs, local_lr, mu, dp_sigma,
                participation=PARTICIPATION, verbose=True):
    torch.manual_seed(RANDOM_SEED)
    gm = TemporalGRU().to(DEVICE)
    gsd = {k: v.detach().clone() for k,v in gm.state_dict().items()}
    n_sel = max(1, int(len(client_rids)*client_ratio))
    best_mcc, best_sd, wait, conv = -1, None, 0, []
    for rnd in range(1, fl_rounds+1):
        by_rsu = {}
        for rid in select_clients(client_rids, n_sel, participation):
            up, sz = client_update(gsd, *client_map[rid], local_epochs, mu, local_lr, dp_sigma)
            by_rsu.setdefault(OBU_TO_RSU[rid], []).append((up, sz))
        rsu_m, rsu_sz, rsu_c = [], [], []
        for r, ups in sorted(by_rsu.items()):
            rsu_m.append(w_fedavg([u for u,_ in ups], [s for _,s in ups]))
            rsu_sz.append(sum(s for _,s in ups)); rsu_c.append(rsu_to_controller(r))
        ctrl_m = []
        for cc in sorted(set(rsu_c)):
            idx = [j for j,c in enumerate(rsu_c) if c==cc]
            ctrl_m.append(w_fedavg([rsu_m[j] for j in idx], [rsu_sz[j] for j in idx]))
        gsd = trimmed_mean(ctrl_m, FL_TRIM_FRAC); gm.load_state_dict(gsd)
        vm = gru_eval(gm, X_va_e, L_va_e, y_va_e)["mcc"]
        conv.append({"round": rnd, "val_mcc": vm, "n_controllers": len(ctrl_m)})
        if vm > best_mcc: best_mcc, best_sd, wait = vm, copy.deepcopy(gsd), 0
        else: wait += 1
        if verbose and rnd % 10 == 0: print(f"  round {rnd:03d}  val MCC={vm:.4f}  best={best_mcc:.4f}")
        if wait >= FL_ES_PATIENCE:
            if verbose: print(f"  early stop round {rnd}");
            break
    if best_sd is not None: gm.load_state_dict(best_sd)
    return best_mcc, gm, pd.DataFrame(conv)

t0 = time.time()
best_val, gru_hier, conv_df = run_fl_hier(**FL_PARAMS)
res = gru_eval(gru_hier, X_te_s, L_te, y_te)
print(f"\n=== 3-tier FL test === MCC={res['mcc']:.4f}  macroF1={res['f1']:.4f}  acc={res['acc']:.4f}  ({time.time()-t0:.0f}s)")
torch.save(gru_hier.state_dict(), os.path.join(MODEL_DIR, "temporal_federated_gru_hier.pt"))
conv_df.to_csv(os.path.join(MODEL_DIR, "fl_hier_convergence.csv"), index=False)
print("saved model + convergence to", MODEL_DIR)

  round 010  val MCC=0.3309  best=0.3309


  round 020  val MCC=0.4118  best=0.4207


  early stop round 24



=== 3-tier FL test === MCC=0.4041  macroF1=0.3332  acc=0.8162  (1066s)
saved model + convergence to /home/sdvn_defense_sybil/35_sybil_attack_project/ns-allinone-3.35/ns-3.35/sybil-attack/notebooks/outputs/temporal_vehicle_tier


## 9. Export the contract parquet

Run the trained model over **all** windows, then pool `phi_temp` and `p_temp` over
observers by snapping `window_start_seconds` to the `EXPORT_GRID_S` grid, so the
output is keyed exactly `(claimed_node_id, window_start_seconds, split)` — the same
key the RSU-tier notebooks and the fusion stage join on.

In [10]:
phi, prob = extract_phi(gru_hier, scale_seq(X_all, L_all, scaler), L_all)  # (N,32),(N,7)
df = meta_df.copy()
df["split"]   = split_arr
df["ws_grid"] = IDM.snap_grid(df["window_start_seconds"])
# attach static attacker-% for the fusion (metadata; not a feature)
_pct = MAN[["run_id","claimed_node_id","attack_percentage"]].drop_duplicates()
df = df.merge(_pct, on=["run_id","claimed_node_id"], how="left")
for i in range(phi.shape[1]):  df[f"phi_temp_{i}"] = phi[:, i]
for k in range(prob.shape[1]): df[f"p_temp_{k}"]   = prob[:, k]

# KEY INCLUDES run_id — the three sims reuse claimed ids, so pooling without run_id
# would average different physical vehicles together (the v3 bug we are fixing).
key = ["run_id", "attack_percentage", "claimed_node_id", "ws_grid", "split"]
val_cols = [f"phi_temp_{i}" for i in range(phi.shape[1])] + [f"p_temp_{k}" for k in range(prob.shape[1])]
agg = {c: "mean" for c in val_cols}; agg["is_sybil"] = "max"
pooled = df.groupby(key, as_index=False).agg(agg)
at = (df.groupby(key)["attack_type"]
        .agg(lambda s: int(pd.Series(s[s>0]).mode().iloc[0]) if (s>0).any() else 0)
        .reset_index())
out = pooled.merge(at, on=key).rename(columns={"ws_grid": "window_start_seconds"})
out = out[["run_id","attack_percentage","claimed_node_id","window_start_seconds",
           "split","is_sybil","attack_type"] + val_cols]
out.to_parquet(OUT_PARQUET, index=False)

json.dump({"model":"temporal_vehicle_tier","window_W":WINDOW_W,"export_grid_s":EXPORT_GRID_S,
           "classes":ATTACK_CLASSES,"features":SEQ_FEATURE_COLS,"runs":[os.path.basename(d) for d in RUN_DIRS],
           "phi_dim":int(phi.shape[1]),"key":key,"test_mcc":float(res["mcc"])},
          open(os.path.join(MODEL_DIR,"config.json"),"w"), indent=2)
print("wrote", OUT_PARQUET, out.shape)
print(out.head(3).to_string())


wrote /home/sdvn_defense_sybil/35_sybil_attack_project/ns-allinone-3.35/ns-3.35/sybil-attack/notebooks/outputs/temporal_vehicle_tier.parquet (49569, 46)
      run_id  attack_percentage  claimed_node_id  window_start_seconds split  is_sybil  attack_type  phi_temp_0  phi_temp_1  phi_temp_2  phi_temp_3  phi_temp_4  phi_temp_5  phi_temp_6  phi_temp_7  phi_temp_8  phi_temp_9  phi_temp_10  phi_temp_11  phi_temp_12  phi_temp_13  phi_temp_14  phi_temp_15  phi_temp_16  phi_temp_17  phi_temp_18  phi_temp_19  phi_temp_20  phi_temp_21  phi_temp_22  phi_temp_23  phi_temp_24  phi_temp_25  phi_temp_26  phi_temp_27  phi_temp_28  phi_temp_29  phi_temp_30  phi_temp_31  p_temp_0  p_temp_1  p_temp_2  p_temp_3  p_temp_4  p_temp_5  p_temp_6
0  pct100_s1                100                8                  48.0  test         0            0   -0.678745    0.991754    0.290347    0.529538    0.817559    0.621069   -0.605397    0.317826    0.780332    0.730401    -0.078350    -0.360893     0.851014    -0.768898